# Apply masks and clean up labkit segmented images

In [ ]:
from pathlib import Path

from bioio import BioImage
import bioio_ome_tiff
import bioio_tifffile
from bioio.writers import OmeTiffWriter

import numpy as np
import pandas as pd

%matplotlib notebook
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

import sys
src_path = str(Path.cwd().parent)
if src_path not in sys.path:
    sys.path.append(src_path)
import src.d00_utils.utilities as utils
import src.d00_utils.dirnames as dn
from src.d01_init_proc import vis_and_rescale as vr
from src.d01_init_proc import subtractbg as sb
from src.d01_init_proc import applymask as am

from scipy import ndimage as ndi
from skimage.measure import label, regionprops

from skimage.filters import threshold_otsu, threshold_multiotsu
from skimage import morphology, segmentation
import numpy.ma as ma

from tqdm import tqdm
from PIL import Image

In [ ]:
input_dirpath = Path(input())

In [ ]:
mask_dirpath = Path(input())

In [ ]:
def create_cmp_fig(seg_cmp, seg_caax, cmp_val=255, caax_val=100):
    cmp_fig = seg_caax * 100 + seg_cmp * 255
    cmp_fig = cmp_fig.astype('uint8').squeeze()
    return cmp_fig

In [ ]:
def subtract_median(img):
    img_ma = np.ma.masked_array(img, img == 0)

    img_ma = np.ma.masked_array(img, img==0)
    median = np.ma.median(img_ma, axis=(3,4))
    median = np.expand_dims(median, axis=(3,4))
    median = np.broadcast_to(median, img.shape).astype('float')
    img_bgsb = img.astype('float') - median
    img_bgsb[img_bgsb < 0] = 0

    return img_bgsb.astype('uint16')

In [ ]:
maskpaths = [p for p in mask_dirpath.glob('*.tif')]
maskpaths.sort()

new_maskdirpath = mask_dirpath.parent / 'other_cellmasks_edited'
new_maskdirpath.mkdir(exist_ok=True)

for maskpath in maskpaths:
    
    mask_file = BioImage(maskpath, reader=bioio_tifffile.Reader)
    mask = mask_file.data
    dtype = mask_file.dtype

    mask = morphology.binary_dilation(mask)
    mask = morphology.binary_dilation(mask)
    
    mask = am.remove_small_holes(mask)

    mask = mask.astype(dtype)
    
    OmeTiffWriter.save(mask, new_maskdirpath / maskpath.name)


In [ ]:
name = maskpath.name
basename = name[:np.strings.find(imgpath.name, 'aligned') + len('aligned')] + '.ome.tif'
basename

In [ ]:
caaxch = 0
cellch = 2
actinch = 3

seg_caaxch = 0
seg_cmpch = 2

figs_dirpath = input_dirpath / dn.figs_dirname
figs_dirpath.mkdir(exist_ok=True, parents=True)

for imgpath in input_dirpath.glob('*.ome.tif'):
    if imgpath.is_file():

        img_file = BioImage(imgpath, reader=bioio_ome_tiff.Reader)
        img = img_file.data

        seg_cmp = (img[:, seg_cmpch, np.newaxis, :, :, :] > 0)
        seg_caax = (img[:, seg_caaxch, np.newaxis, :, :, :] > 0)
        seg_cell = seg_cmp | seg_caax

        name = imgpath.name
        basename = name[:np.strings.find(imgpath.name, 'aligned') + len('aligned')] + '.ome.tif'
        orig_imgpath = orig_dirpath / basename
        print(orig_imgpath.is_file())

        orig_img_file = BioImage(orig_imgpath, reader=bioio_ome_tiff.Reader)
        orig_img = orig_img_file.data


        # #EDIT SEGMENTATION

        # # edit cell seg
        # seg_cell = am.mask_img(seg_cell, mask)
        # seg_cell = am.remove_unconnected_areas(seg_cell)
        # seg_cell = am.remove_small_holes(seg_cell, area_threshold=5)
        
        # # edit caax seg
        # seg_caax = am.mask_img(seg_caax, seg_cell)
        # seg_caax = am.remove_small_holes(seg_caax, area_threshold=5)
    
        # # edit cmp seg
        # seg_cmp = (seg_cell & ~seg_caax)
        # temp_seg_cell = am.remove_small_holes(seg_cell, area_threshold=50)
        # seg_cmp = am.remove_cmp_near_cellborders(temp_seg_cell, seg_cmp, iter_dil=3)

        # # edit cell seg one more time and use it to mask caax and cmp
        # seg_cell = seg_cmp | seg_caax
        # seg_cell = am.remove_unconnected_areas(seg_cell)
        # seg_cmp = seg_cmp & seg_cell
        # seg_caax = seg_caax & seg_cell

        # stack = np.concatenate([img[:, [caaxch, cellch], :, :, :], actin, seg_cmp, seg_caax], axis=1).astype('uint16')

        # stack[:, 2:, :, :, :] = am.mask_img(stack[:, 2:, :, :, :], mask)

        # ROIname = maskpath.name.replace('.tif', '.ome.tif')
        # savepath = output_dirpath / ROIname
        # ome_metadata = utils.construct_ome_metadata(stack, img_file)
        # OmeTiffWriter.save(stack, savepath, ome_xml=ome_metadata)

        # cmp_fig = vr.create_cmp_fig(seg_cmp, seg_caax)
        # OmeTiffWriter.save(cmp_fig, figs_dirpath / imgpath.name)

In [ ]:
# resave manually edited images
manually_edited_dirpath = Path(input())

In [ ]:
imgpaths = [path for path in manually_edited.glob('*edited.ome.tif')]
print(len(imgpaths))

for imgpath in tqdm(imgpaths):
    img = BioImage(imgpath, reader=bioio_tifffile.Reader).data   
    ome_metadata = utils.construct_ome_metadata(img, img_file)
    OmeTiffWriter.save(img, imgpath, ome_xml=ome_metadata)

In [ ]:
caax_cell_actin_seg_edited_dirpath = manually_edited_dirpath.parent
older_dirpath = caax_cell_actin_seg_edited_dirpath / 'older'
for imgpath in imgpaths:
    prev_imgname = imgpath.name.replace("_edited", "")
    older_imgpath = caax_cell_actin_seg_edited_dirpath / prev_imgname
    
    if older_imgpath.is_file():
        older_imgpath.rename(older_dirpath / prev_imgname)

    imgpath.rename(caax_cell_actin_seg_edited_dirpath / prev_imgname)
        

In [ ]:
df_path = Path(input())

In [ ]:
df = pd.read_csv(df_path)
df.head()

In [ ]:
names = df['image name'].str.replace('-', '_')
namesplits = names.str.split('_')
print(namesplits)
df['wellID'] = namesplits.str[0] + '_' + namesplits.str[1] + '_' + namesplits.str[6]

sns.scatterplot(df, x='wellID', y='mean_int_ch0')
plt.show()


#sns.scatterplot(df, x='mean_int_ch0', y='% exclusion')